In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

In [ ]:
max_seq_length = 1024
dtype = None
load_in_4bit = True

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj","down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,

)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("Prarabdha/indian-legal-supervised-fine-tuning-data",split="train[:20000]")

In [ ]:
dataset[:2]

In [ ]:
legal_prompt = """You are a highly knowledgeable Indian legal assistant trained on court judgments, case law, and statutory interpretation.

Your task is to carefully read the legal context and answer the question precisely, using formal legal reasoning.

### Legal Context:
{0}

### Question:
{1}

### Answer:
{2}"""
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    contexts = examples["context"]
    questions = examples["question"]
    responses = examples["response"]

    texts = []

    for context, question, response in zip(contexts, questions, responses):
        text = legal_prompt.format(context, question, response) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}

dataset = dataset.map(formatting_prompts_func,batched=True,)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported
import torch

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        max_steps = 500,              # Increased from 60 (important)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
from unsloth.chat_templates import get_chat_template
from unsloth import FastLanguageModel
import torch

# Enable faster inference
FastLanguageModel.for_inference(model)

# Apply Llama 3.1 chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# Legal reasoning question
legal_question = """
A owns a piece of land which is under a long-standing boundary dispute with his neighbor B.
One day, A removes certain movable property (wooden fencing panels) from the disputed boundary area believing in good faith that the property belongs to him.

B files a complaint alleging theft under Section 379 of the Indian Penal Code.

If A genuinely believed that he had a lawful claim over the property and lacked dishonest intention, can he be convicted under Section 379 IPC?

Discuss with reference to the essential ingredients of theft and the concept of “claim of right.”
"""

messages = [
    {
        "role": "user",
        "content": legal_question
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Generate answer
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    temperature = 0.7,
    top_p = 0.9,
    use_cache = True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

In [ ]:
model.save_pretrained_merged("legal_full_model", tokenizer)

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model.push_to_hub("Kevinzyyy/legal-llm")
tokenizer.push_to_hub("Kevinzyyy/legal-llm")